## Cell 1: Environment Setup & Library Imports

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, shutil, glob, re, time
import torch
import numpy as np
import pandas as pd
import torch.nn as nn

from torch.utils.data import Dataset, DataLoader
from torch.ao.quantization import quantize_dynamic
from torchvision import transforms
from PIL import Image
from sklearn.model_selection import GroupShuffleSplit

## Cell 2: Path Definitions & Dataset Preparation

In [ ]:
DATASETS_DIR     = '/content/drive/MyDrive/datasets'  # Update if using a different directory
LOCAL_BASE       = '/tmp/datasets'
SAVE_WEIGHTS_DIR = os.path.join(os.path.dirname(DATASETS_DIR), 'checkpoints')

if not os.path.exists(LOCAL_BASE):
    print('Copying dataset from Drive to local SSD...')
    shutil.copytree(DATASETS_DIR, LOCAL_BASE)

AFRICAN_TUMOR_DIR   = os.path.join(LOCAL_BASE, 'braTS_africa_test', 'glioma')
AFRICAN_HEALTHY_DIR = os.path.join(LOCAL_BASE, 'nigerian_healthy', 'axial')
SPLIT_RATIO         = 0.40


class MRIDataset(Dataset):
    def __init__(self, filepaths, labels, transform=None):
        self.filepaths = filepaths
        self.labels    = labels
        self.transform = transform

    def __len__(self):
        return len(self.filepaths)

    def __getitem__(self, idx):
        img = Image.open(self.filepaths[idx]).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, torch.tensor(self.labels[idx], dtype=torch.float32)


val_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])


def extract_patient_id(filename):
    base = os.path.basename(filename)
    return base.split('_slice')[0] if '_slice' in base else base.split('.')[0]


def scan_dir(directory, label):
    filepaths, labels, groups = [], [], []
    if os.path.exists(directory):
        for ext in ('*.png', '*.jpg', '*.jpeg', '*.PNG', '*.JPG', '*.JPEG'):
            for f in sorted(glob.glob(os.path.join(directory, '**', ext), recursive=True)):
                filepaths.append(f)
                labels.append(label)
                groups.append(extract_patient_id(f))

    df = pd.DataFrame({'filepath': filepaths, 'label': labels, 'patient_id': groups})
    return df.sort_values(by=['patient_id', 'filepath']).reset_index(drop=True)


df_afr_tumor   = scan_dir(AFRICAN_TUMOR_DIR,   label=1)
df_afr_healthy = scan_dir(AFRICAN_HEALTHY_DIR, label=0)

## Cell 3: Full Architecture Pipeline Definition

In [ ]:
class DINOv2Classifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.classifier = nn.Sequential(
            nn.Linear(384, 128),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(128, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.classifier(x).squeeze(-1)


class FullDINOv2Pipeline(nn.Module):
    def __init__(self, backbone, head):
        super().__init__()
        self.backbone = backbone
        self.head     = head

    def forward(self, x):
        with torch.no_grad():
            return self.head(self.backbone(x))

## Cell 4: INT8 Dynamic Quantization & Benchmark Evaluation (Table III)

In [ ]:
device_cpu = torch.device('cpu')
SEED_QUANT = 2026


def print_formatted_table(title, df):
    headers = list(df.columns)
    data    = df.values.tolist()
    widths  = [
        max(len(str(h)), max(len(str(row[i])) for row in data))
        for i, h in enumerate(headers)
    ]
    border     = '─' * (sum(widths) + 3 * (len(widths) - 1))
    header_str = '   '.join(
        f'{headers[i]:<{widths[i]}}' if i == 0 else f'{headers[i]:>{widths[i]}}'
        for i in range(len(headers))
    )
    print(title)
    print(border)
    print(header_str)
    print(border)
    for row in data:
        print('   '.join(
            f'{str(row[i]):<{widths[i]}}' if i == 0 else f'{str(row[i]):>{widths[i]}}'
            for i in range(len(row))
        ))
    print(border)


print('\nRunning INT8 dynamic quantization benchmark (Seed 2026)...\n')

gss_2026  = GroupShuffleSplit(n_splits=1, test_size=SPLIT_RATIO, random_state=SEED_QUANT)
_, test_idx_2026 = next(gss_2026.split(df_afr_tumor, groups=df_afr_tumor['patient_id']))
df_holdout_2026  = df_afr_tumor.iloc[test_idx_2026].reset_index(drop=True)

holdout_loader_2026 = DataLoader(
    MRIDataset(df_holdout_2026['filepath'].values, df_holdout_2026['label'].values, val_transforms),
    batch_size=32, shuffle=False
)

healthy_loader = DataLoader(
    MRIDataset(df_afr_healthy['filepath'].values, df_afr_healthy['label'].values, val_transforms),
    batch_size=32, shuffle=False
)

backbone_quant = torch.hub.load('facebookresearch/dinov2', 'dinov2_vits14').to(device_cpu)
backbone_quant.eval()
for p in backbone_quant.parameters():
    p.requires_grad = False

head_quant = DINOv2Classifier().to(device_cpu)
head_quant.load_state_dict(torch.load(
    os.path.join(SAVE_WEIGHTS_DIR, f'dinov2_seed_{SEED_QUANT}.pth'), map_location=device_cpu
))
head_quant.eval()

full_model_fp32 = FullDINOv2Pipeline(backbone_quant, head_quant).to(device_cpu).eval()

# Dynamic quantization targeting all 50 Linear layers (48 backbone + 2 head)
full_model_int8 = quantize_dynamic(full_model_fp32, {nn.Linear}, dtype=torch.qint8).to(device_cpu).eval()

fp32_path = os.path.join(SAVE_WEIGHTS_DIR, 'full_model_fp32.pth')
int8_path = os.path.join(SAVE_WEIGHTS_DIR, 'full_model_int8.pth')
torch.save(full_model_fp32.state_dict(), fp32_path)
torch.save(full_model_int8.state_dict(), int8_path)
fp32_size_mb = os.path.getsize(fp32_path) / (1024 * 1024)
int8_size_mb = os.path.getsize(int8_path) / (1024 * 1024)
size_ratio   = fp32_size_mb / int8_size_mb

torch.set_num_threads(1)
dummy_input = torch.randn(1, 3, 224, 224, device=device_cpu)


def measure_latency(model, sample_input, num_warmup=20, num_runs=30):
    with torch.no_grad():
        for _ in range(num_warmup):
            model(sample_input)
        latencies = []
        for _ in range(num_runs):
            t0 = time.perf_counter()
            model(sample_input)
            latencies.append((time.perf_counter() - t0) * 1000.0)
    return np.median(latencies)


fp32_latency_ms = measure_latency(full_model_fp32, dummy_input)
int8_latency_ms = measure_latency(full_model_int8, dummy_input)
speedup_ratio   = fp32_latency_ms / int8_latency_ms
throughput_fp32 = 1000.0 / fp32_latency_ms
throughput_int8 = 1000.0 / int8_latency_ms

torch.set_num_threads(os.cpu_count() or 4)


def evaluate_metrics(model, tumor_ld, healthy_ld):
    model.eval()
    tumor_probs, healthy_probs = [], []
    with torch.no_grad():
        for imgs, _ in tumor_ld:
            tumor_probs.extend(model(imgs.to(device_cpu)).squeeze(-1).cpu().numpy())
        for imgs, _ in healthy_ld:
            healthy_probs.extend(model(imgs.to(device_cpu)).squeeze(-1).cpu().numpy())
    tumor_probs   = np.array(tumor_probs)
    healthy_probs = np.array(healthy_probs)
    return (tumor_probs >= 0.5).mean() * 100.0, (healthy_probs < 0.5).mean() * 100.0


fp32_sens, fp32_spec = evaluate_metrics(full_model_fp32, holdout_loader_2026, healthy_loader)
int8_sens, int8_spec = evaluate_metrics(full_model_int8, holdout_loader_2026, healthy_loader)


def fmt_change(val):
    return '0.00' if abs(val) < 1e-6 else f'{val:+.2f}'


df_table3_fmt = pd.DataFrame([
    {'Metric': 'Model size (MB)',       'FP32': f'{fp32_size_mb:.2f}',    'INT8': f'{int8_size_mb:.2f}',    'Change': f'{size_ratio:.2f}× smaller'},
    {'Metric': 'Median latency (ms)',   'FP32': f'{fp32_latency_ms:.1f}', 'INT8': f'{int8_latency_ms:.1f}', 'Change': f'{speedup_ratio:.2f}× faster'},
    {'Metric': 'Throughput (images/s)', 'FP32': f'{throughput_fp32:.2f}', 'INT8': f'{int8_latency_ms:.2f}', 'Change': f'{throughput_int8/throughput_fp32:.2f}× higher'},
    {'Metric': 'Sensitivity (%)',       'FP32': f'{fp32_sens:.2f}',       'INT8': f'{int8_sens:.2f}',       'Change': fmt_change(int8_sens - fp32_sens)},
    {'Metric': 'Specificity (%)',       'FP32': f'{fp32_spec:.2f}',       'INT8': f'{int8_spec:.2f}',       'Change': fmt_change(int8_spec - fp32_spec)},
])
print_formatted_table('TABLE III.  FP32 Baseline vs. INT8 Dynamic Quantization (Seed 2026)', df_table3_fmt)

## Cell 5: Export Test Images for Raspberry Pi Edge Evaluation

In [ ]:
SEED_EXPORT     = 2026
OUTPUT_TEST_DIR = os.path.join(DATASETS_DIR, 'test_dataset_seed2026')
TUMOUR_OUT_DIR  = os.path.join(OUTPUT_TEST_DIR, 'tumour')
HEALTHY_OUT_DIR = os.path.join(OUTPUT_TEST_DIR, 'healthy')

if os.path.exists(OUTPUT_TEST_DIR):
    shutil.rmtree(OUTPUT_TEST_DIR)
os.makedirs(TUMOUR_OUT_DIR,  exist_ok=True)
os.makedirs(HEALTHY_OUT_DIR, exist_ok=True)

gss = GroupShuffleSplit(n_splits=1, test_size=SPLIT_RATIO, random_state=SEED_EXPORT)
_, test_idx = next(gss.split(df_afr_tumor, groups=df_afr_tumor['patient_id']))
df_holdout_2026 = df_afr_tumor.iloc[test_idx].reset_index(drop=True)

for src_path in df_holdout_2026['filepath']:
    shutil.copy2(src_path, os.path.join(TUMOUR_OUT_DIR, os.path.basename(src_path)))

for src_path in df_afr_healthy['filepath']:
    shutil.copy2(src_path, os.path.join(HEALTHY_OUT_DIR, os.path.basename(src_path)))

print(f'Tumour slices exported:  {len(df_holdout_2026)} (seed-2026 holdout, 38 patients)')
print(f'Healthy slices exported: {len(df_afr_healthy)}')

zip_base = os.path.join(DATASETS_DIR, 'test_dataset_seed2026')
shutil.make_archive(zip_base, 'zip', OUTPUT_TEST_DIR)
print(f'Archive ready: {zip_base}.zip')